In [99]:
%matplotlib inline

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown, FloatLogSlider, HTML
from IPython.display import display, clear_output, Audio
import sys
import PathGeneration as pg
from Interpolation import SplineTrajectory3D

build_dir = "cppSimulation/build"
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)

import propagationmod
import resamplemod

In [111]:
# audio utils

def db_scale(x, amin=1e-10):
    return 20.0 * np.log10(np.maximum(x, amin))

def lin_scale(x):
    return 10**(x / 20)

def extend_noise(x, target_samples):
    if len(x) < target_samples: 
        X = np.fft.rfft(x)
        X *= np.exp(1j*np.random.rand(len(X))*2*np.pi)
        x = np.tile(np.fft.irfft(X), (target_samples-1)//len(x)+1)
    return x[:target_samples]

def extend_audio(x, target_samples):
    if len(x) >= target_samples:
        return x[: target_samples]
    fade = 2**11
    n = len(x)
    window = np.minimum(np.minimum(np.arange(n)/fade, n/fade-np.arange(n)/fade), 1)
    grain = x * (0.5 - 0.5 * np.cos(np.pi*window))
    y = np.zeros(((target_samples-1)//n+2)*n + 4 * fade)
    for i in range(0, len(y)-n-1, n-fade):
        y[i:i+n] += grain
    return y[fade:fade+target_samples]

def get_audio_from_source(source):
    if isinstance(source, str):
        data, sr = sf.read(source, always_2d=True)
        if data.size == 0:
            raise RuntimeError("Empty or unsupported audio file.")
        audio = data[:, 0]
        return (audio, sr)
    elif isinstance(source, tuple):
        return source
    else:
        raise TypeError("bad scene audio type")
    
def propagate_audio(position_data, mic_positions, audio, sr):
    if isinstance(position_data, tuple):
        pos, vel = position_data
        return propagationmod.simulate_audio_propagation(sr, mic_positions, pos, vel, audio)
    else:
        return propagationmod.static_audio_propagation(sr, mic_positions, position_data, audio)

def rms_normalize(x):
    return x / np.mean(x * x)

def prepare_audio(audio_source, mic_array, position, total_samples, sr):
    
    audio, sr2 = get_audio_from_source(audio_source)
    
    if sr2 != sr:
        audio = resamplemod.sinc_resample(audio, sr/sr2, 16)
    
    audio = rms_normalize(audio)
    audio = extend_audio(audio, total_samples)
    audio = propagate_audio(position, mic_array, audio, sr)

    return audio

def create_uncorrelated_noise(m, n):
    noise = np.random.randn(m, n)
    for i in range(m):
        noise[i] = rms_normalize(noise[i])
    return noise

def create_diffuse_noise(n, micpos, sr):
    m = micpos.shape[1]
    noise = np.zeros((m, n), dtype=np.float32)
    vel = np.zeros((3, n), dtype=np.float32)
    for _ in range(2):
        pos = np.random.randn(3, n)
        pos /= np.linalg.norm(pos, axis=0)
        noise += propagationmod.simulate_audio_propagation(sr, micpos, pos, vel, np.random.randn(n))
    for i in range(m):
        noise[i] = rms_normalize(noise[i])
    return noise

In [112]:
# scene utils

class Scene:
    target_position = ()
    interferer_position = []
    interferer_audio = []

def prepare_position(position, t):
    if isinstance(position, tuple):
        pos, t2 = position
        return SplineTrajectory3D(t2, pos).get(t)
    return position

def create_prepared_scene(scene, duration, sr):
    prepared_scene = Scene()
    t = np.linspace(0, duration, duration*sr)
    prepared_scene.target_position = prepare_position(scene.target_position, t)
    prepared_scene.interferer_position = [prepare_position(i, t) for i in scene.interferer_position]
    prepared_scene.interferer_audio = [get_audio_from_source(i) for i in scene.interferer_audio]
    return prepared_scene

def scene1(duration, sr):
    scene = Scene()
    scene.target_position = np.array([5.0, 5.0, 5.0])
    return create_prepared_scene(scene, duration, sr)

In [113]:
# plot utils

def plot_3d_base(ax, azim, elev, size=7.5):
    """
    Plot a 3D flight path and a reference point at the origin.
    points: numpy array of shape (3, N)
    t: numpy array of shape (N,)
    """
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.view_init(elev=elev, azim=azim)

    ax.grid(True)

    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_zlim(0, 2 * size)

def plot_dot_3d(ax, point, color):
    ax.scatter(*point, color=color, s=100)

def plot_path_3d(ax, points, t_idx, color):
    resolution = 2**12
    step = max(1, points.shape[1] // resolution)
    idx = np.arange(0, points.shape[1], step)
    if points.shape[1] % step != 0:
        idx = np.append(idx, [points.shape[1]-1]) 
    p = points[:,idx]
    x, y, z = p

    ax.plot(x, y, z, color=color)
    plot_dot_3d(ax, points[:,t_idx], color)

def plot_position_3d(ax, position, t_idx, color):
    if isinstance(position, tuple):
        plot_path_3d(ax, position[0], t_idx, color)
    else:
        plot_dot_3d(ax, position, color)

In [121]:
total_duration = 10
fig_width = 11
dpi = 80
sr = 48000

microphone_array_pos = np.array(
    [
        [-0.1, -0.1, 0],
        [-0.1,  0.1, 0],
        [ 0.1, -0.1, 0],
        [ 0.1,  0.1, 0],
    ]
).T

m = microphone_array_pos.shape[1]
n = total_duration * sr

audio_sources = {
    "mavic4":  "../samples/mavic4_1s.wav",
    "noise": np.random.randn(n)
}

scenes = {
    "scene 1": scene1
}

class AudioScene:
    scene: Scene = None
    target = np.zeros((m, n), dtype=np.float32)
    noise = create_uncorrelated_noise(m, n)
    diffuse = create_diffuse_noise(n, microphone_array_pos, sr)
    interferer = np.zeros((m, n), dtype=np.float32)
    x = np.zeros((m, n), dtype=np.float32)
audio = AudioScene()



w_target = Dropdown(options=[*audio_sources.keys()], value=[*audio_sources.keys()][0], description="Target:")
w_scene = Dropdown(options=[*scenes.keys()], value=[*scenes.keys()][0], description="Scene:")
w_NFFT = Dropdown(options=[2**i for i in range(6, 14)], value=1024, description="NFFT:")
w_algo = Dropdown(options=["DAS", "MVDR", "MUSIC"], value="DAS", description="Algorithm:")

w_time = FloatSlider(
    value=0.0, min=0.0, max=max(0.0, total_duration - 0.01),
    step=max(0.001, total_duration / 1000.0),
    description="Time (s)", continuous_update=True, readout_format=".2f"
)
w_azi = FloatSlider(
    value=30, min=-180, max=180, step=1,
    description="Azi (deg)", continuous_update=True, readout_format=".0f"
)
w_ele = FloatSlider(
    value=30, min=-90, max=90, step=1,
    description="Ele (deg)", continuous_update=True, readout_format=".0f"
)



w_noise_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Noise (dB)", continuous_update=True, readout_format=".1f"
)
w_diffuse_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Diffuse (dB)", continuous_update=True, readout_format=".1f"
)
w_interferer_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Interf. (dB)", continuous_update=True, readout_format=".1f"
)
w_target_db = FloatSlider(
    value=0, min=-80, max=20, step=0.1,
    description="Target (dB)", continuous_update=True, readout_format=".1f"
)
for i in [w_time, w_azi, w_ele, w_noise_db, w_diffuse_db, w_interferer_db, w_target_db]:
    i.layout.width = f"{fig_width * dpi * 0.6}px"



out1 = Output()
out2 = Output()

def update_plot(change=None):
    time = max(0.0, min(w_time.value, total_duration))
    t_idx = int(round(time * sr))
    t_idx = max(0, min(t_idx, total_duration-1))

    with out1:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width*0.4, fig_width*0.3))
        ax = fig.add_subplot(111, projection='3d')
        plot_3d_base(ax, w_azi.value, w_ele.value)
        plot_position_3d(ax, audio.scene.target_position, t_idx, 'blue')
        plot_dot_3d(ax, microphone_array_pos, 'red')
        plt.tight_layout()
        plt.show()

    with out2:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width*0.4, fig_width*0.3))
        ax = fig.add_subplot(111, projection='3d')
        plot_3d_base(ax, w_azi.value, w_ele.value)
        plot_position_3d(ax, audio.scene.target_position, t_idx, 'blue')
        plot_dot_3d(ax, microphone_array_pos, 'red')
        plt.tight_layout()
        plt.show()

def update_audio(change=None, update_plt=True):
    g_n = lin_scale(w_noise_db.value)
    g_t = lin_scale(w_target_db.value)
    g_i = lin_scale(w_interferer_db.value)
    g_d = lin_scale(w_diffuse_db.value)
    audio.x = audio.target * g_t + audio.noise * g_n # + audio.interferer * g_i + audio.diffuse * g_d
    if update_plt:
        update_plot()

def update_scene(change=None, update_plt=True):
    scene = scenes[w_scene.value](total_duration, sr)
    audio.scene = scene
    audio.target = prepare_audio(audio_sources[w_target.value], microphone_array_pos, scene.target_position, n, sr)
    audio.interferer = np.zeros((m, n), dtype=np.float32)
    for i in range(len(scene.interferer_audio)):
        audio.interferer += prepare_audio(
            scene.interferer_audio[i], 
            microphone_array_pos, 
            scene.interferer_position[i], n, sr)
    update_audio(update_plt=update_plt)
    
for w in (w_target, w_scene):
    w.observe(update_scene, names="value")

for w in (w_time, w_azi, w_ele, w_NFFT, w_algo):
    w.observe(update_plot, names="value")

for w in (w_target_db, w_noise_db, w_diffuse_db, w_interferer_db):
    w.observe(update_audio, names="value")

display(VBox([
    HBox([w_target, w_scene, w_algo, w_NFFT]),
    HBox([
        VBox([w_time, w_ele, w_azi,
              Label("Noise levels"),
              w_target_db, w_noise_db, w_interferer_db, w_diffuse_db
              ]),
        out1
    ]),
    out2
]))

update_scene(update_plt=False)
update_plot()